<div style="
background-color:#EAEAEA;
padding:15px;
border-left:5px solid #6C757D;
border-radius:6px;">

# Master's Thesis in Advanced Physics
---

This notebook is part of the **Master's Thesis (MSc Dissertation)**: **Fast Simulation of Neutrino Oscillations in Matter**.

**Author**  
Juan Ramon Diaz Santos <diazjuan@alumni.uv.es>

**Supervisors**  
Roberto Ruiz de Austri Bazan <rruiz@ific.uv.es>  
Michele Lucente <michele.lucente@unibo.it>

**Date**  
June 2026
</div>

# Core Diagnostic 2 - Propagation, Probability, and Flux
---

This notebook diagnoses the medium-independent propagation utilities in `tpeanuts.core.common`: generic evolutor composition, probability projection, and flux construction. The automated checks live in `tpeanuts.core.common.test.test4_evolutor`, `tpeanuts.core.common.test.test5_probability`, and `tpeanuts.core.common.test.test6_flux`; here we keep the plots and compact numerical diagnostics.

---

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: evolution operators, probabilities, and fluxes |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Generic-Evolutor) | **Generic Evolutor**: state evolution and segment composition |
| [4](#4.-Probability) | **Probability**: coherent and incoherent projections |
| [5](#5.-Flux) | **Flux**: normalization, spectrum, and flavour-resolved flux |
| [?](#6.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 Evolution Operators

For a coherent neutrino state in flavour space,

$$
|\nu(L)\rangle = S(L)|\nu(0)\rangle,
$$

where $S$ is a $3\times3$ evolution operator. If propagation is split into ordered segments, the total operator is a matrix product. With the convention used in `compose_segment_evolutors`, `multiply="left"` reproduces the historical accumulation

$$
S_{\rm total}=S_N\cdots S_2S_1,
$$

where each new segment acts on the left.

### 0.2 Oscillation Probabilities

Given a flavour-basis evolutor $S$, transition probabilities are defined as

$$
P(\nu_\alpha\to\nu_\beta)=|S_{\beta\alpha}|^2.
$$

The initial flavour labels the column, the final flavour labels the row. For a unitary evolutor, the probability matrix is column-normalized:

$$
\sum_\beta P_{\beta\alpha}=1.
$$

### 0.3 Coherent and Incoherent Initial States

A coherent state is projected by applying $S$ to amplitudes and taking component moduli. An incoherent source is propagated by applying the probability matrix to weights,

$$
w_\beta^{\rm final}=\sum_\alpha P_{\beta\alpha}w_\alpha^{\rm initial}.
$$

For incoherent mass-basis weights, the implementation uses the amplitudes $S U$, where $U$ is the PMNS matrix.

### 0.4 Flux Construction

The flux helper is deliberately model-neutral. Once final flavour probabilities are known, a flux normalization $\Phi$ and optional spectral weight $f(E)$ are applied as

$$
\Phi_\beta(E)=P_\beta(E)\,\Phi\,f(E).
$$

The helper only handles broadcasting and scaling; it does not compute probabilities or source spectra.

### References

- C. Giunti and C. W. Kim, *Fundamentals of Neutrino Physics and Astrophysics*, Oxford University Press, 2007.
- S. M. Bilenky, *Introduction to the Physics of Massive and Mixed Neutrinos*, Springer, 2010.

## 1. Libraries

All imports are centralized here. The notebook uses simple analytic rotations so the diagnostics stay independent of medium geometry and external data files.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import torch

from tpeanuts.core.common.evolutor import apply_evolutor_to_state, compose_segment_evolutors
from tpeanuts.core.common.flux import flux_from_probability
from tpeanuts.core.common.probability import (
    check_probability_conservation,
    check_probability_matrix,
    normalize_probability_columns,
    probability_coherent,
    probability_coherent_state,
    probability_from_evolutor,
    probability_incoherent,
    probability_transition,
)
from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import save_and_show, to_numpy
from tpeanuts.util.context import RuntimeContext

## 2. Paths and Configuration

### 2.1 Paths

Figures are saved under the notebook-relative output path `diagnostic/core`.

In [ ]:
config = load_notebook_config()
OUTPUT_DIR = config.output_dir("diagnostic", "core")
SHOW_PLOTS = config.show_plots

print(f"Repository root: {config.package_dir}")
print(f"Figure output directory: {OUTPUT_DIR}")

### 2.2 Configuration

We use rotations in the $(\nu_e,\nu_\mu)$ plane as exactly unitary toy evolutors. This gives closed-form expectations while still exercising the same tensor paths as realistic propagation.

**Expected results**: probability columns should sum to one, segment products should remain unitary, and normalized probabilities should preserve total flux after scaling.

In [ ]:
ctx = RuntimeContext.resolve(config.device, config.dtype)
CDTYPE = torch.complex128 if ctx.dtype == torch.float64 else torch.complex64

THETA_GRID = torch.linspace(0.0, 1.2, 121, device=ctx.device, dtype=ctx.dtype)
SEGMENT_THETAS = torch.tensor([0.12, 0.23, -0.08, 0.31], device=ctx.device, dtype=ctx.dtype)
INITIAL_STATE_E = torch.tensor([1.0 + 0.0j, 0.0 + 0.0j, 0.0 + 0.0j], device=ctx.device, dtype=CDTYPE)
INITIAL_WEIGHTS = torch.tensor([0.65, 0.25, 0.10], device=ctx.device, dtype=ctx.dtype)
ENERGY_GRID = torch.logspace(-1, 1, 120, device=ctx.device, dtype=ctx.dtype)
SOURCE_FLUX = torch.tensor(5.0, device=ctx.device, dtype=ctx.dtype)

print("theta grid:", tuple(THETA_GRID.shape))
print("segment angles:", SEGMENT_THETAS)
print("energy grid [a.u.]:", float(ENERGY_GRID[0]), "to", float(ENERGY_GRID[-1]))

### 2.3 Local Helpers

The only helper builds the same rotation family used by the tests.

In [ ]:
def rotation(theta):
    c = torch.cos(theta)
    s = torch.sin(theta)
    zeros = torch.zeros_like(theta)
    ones = torch.ones_like(theta)
    rows = [
        torch.stack([c, s, zeros], dim=-1),
        torch.stack([-s, c, zeros], dim=-1),
        torch.stack([zeros, zeros, ones], dim=-1),
    ]
    return torch.stack(rows, dim=-2).to(dtype=CDTYPE)


def heatmap(matrix, title, filename, *, cmap="viridis", vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    im = ax.imshow(to_numpy(matrix), cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xlabel("initial flavour")
    ax.set_ylabel("final flavour")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    save_and_show(filename, fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)

## 3. Generic Evolutor

### 3.1 State Evolution Scan

A pure $\nu_e$ state is evolved with rotations of increasing angle. The first two components exchange probability while the third remains unchanged.

**Expected results**: $P_e+P_\mu=1$ and $P_\tau=0$ for every angle in this toy model.

In [ ]:
U_scan = rotation(THETA_GRID)
states_scan = apply_evolutor_to_state(U_scan, INITIAL_STATE_E)
prob_scan = probability_coherent_state(states_scan)

fig, ax = plt.subplots(figsize=(7.8, 4.2))
labels = [r"$P_e$", r"$P_\mu$", r"$P_\tau$"]
for i, label in enumerate(labels):
    ax.plot(to_numpy(THETA_GRID), to_numpy(prob_scan[:, i]), lw=1.8, label=label)
ax.set_xlabel(r"rotation angle $\theta$")
ax.set_ylabel("component probability")
ax.set_title("Coherent state evolution under a toy unitary rotation")
ax.legend()
fig.tight_layout()
save_and_show("diagnostic2_fig3_1_state_evolution_scan.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max |sum(P)-1|:", torch.max(torch.abs(prob_scan.sum(dim=-1) - 1.0)).item())

### 3.2 Segment Composition

Segment operators are composed into a total operator. This checks the product convention used by `compose_segment_evolutors`.

**Expected results**: composing rotations should give another rotation with the sum of the segment angles, and the product should remain unitary.

In [ ]:
segments = rotation(SEGMENT_THETAS)
U_total = compose_segment_evolutors(segments, multiply="right")
U_expected = rotation(SEGMENT_THETAS.sum())
unitarity = U_total.conj().T @ U_total - torch.eye(3, device=ctx.device, dtype=CDTYPE)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True)
for ax, matrix, title in zip(
    axes,
    [torch.abs(U_total - U_expected), torch.abs(unitarity)],
    [r"$|U_{total}-U(\sum\theta_i)|$", r"$|U^\dagger U-I|$"],
):
    im = ax.imshow(to_numpy(matrix), cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

save_and_show("diagnostic2_fig3_2_segment_composition.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max composition residual:", torch.max(torch.abs(U_total - U_expected)).item())
print("max unitarity residual:", torch.max(torch.abs(unitarity)).item())

## 4. Probability

### 4.1 Transition Probability Matrix

The full probability matrix follows $P_{\beta\alpha}=|S_{\beta\alpha}|^2$.

**Expected results**: the matrix is non-negative and column-normalized for a unitary evolutor.

In [ ]:
U_demo = rotation(torch.tensor(0.45, device=ctx.device, dtype=ctx.dtype))
P_demo = probability_transition(U_demo)
heatmap(P_demo, "Transition probability matrix", "diagnostic2_fig4_1_probability_matrix.png", vmin=0.0, vmax=1.0)
print("column sums:", P_demo.sum(dim=-2))
print("conservation check:", check_probability_conservation(P_demo))
print("matrix check:", check_probability_matrix(P_demo))

### 4.2 Coherent versus Incoherent Projection

A coherent state keeps amplitude phases before projection. An incoherent state applies the probability matrix directly to classical weights.

**Expected results**: coherent and incoherent outputs differ in general because they represent different physical inputs, but both remain normalized when the input is normalized.

In [ ]:
coherent_state = torch.tensor([1.0, 1.0j, 0.0], device=ctx.device, dtype=CDTYPE) / torch.sqrt(torch.tensor(2.0, device=ctx.device, dtype=ctx.dtype))
coherent_prob = probability_coherent(U_demo, coherent_state)
incoherent_prob = probability_incoherent(P_demo, INITIAL_WEIGHTS)

fig, ax = plt.subplots(figsize=(7.2, 4.0))
x = np.arange(3)
width = 0.36
ax.bar(x - width / 2, to_numpy(coherent_prob), width=width, label="coherent amplitudes")
ax.bar(x + width / 2, to_numpy(incoherent_prob), width=width, label="incoherent weights")
ax.set_xticks(x)
ax.set_xticklabels([r"$\nu_e$", r"$\nu_\mu$", r"$\nu_\tau$"])
ax.set_ylabel("final probability / weight")
ax.set_title("Coherent and incoherent projection paths")
ax.legend()
fig.tight_layout()
save_and_show("diagnostic2_fig4_2_coherent_incoherent.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("sum coherent:", coherent_prob.sum().item())
print("sum incoherent:", incoherent_prob.sum().item())

### 4.3 Normalization Repair Diagnostic

`normalize_probability_columns` is an explicit repair tool for matrices that are close to, but not exactly, column-normalized.

**Expected results**: the repaired matrix has column sums equal to one.

In [ ]:
P_perturbed = P_demo.clone()
P_perturbed[0, 0] += 0.05
P_perturbed[1, 1] *= 0.90
P_repaired = normalize_probability_columns(P_perturbed)

fig, ax = plt.subplots(figsize=(7.2, 4.0))
columns = np.arange(3)
ax.plot(columns, to_numpy(P_perturbed.sum(dim=-2)), marker="o", label="before")
ax.plot(columns, to_numpy(P_repaired.sum(dim=-2)), marker="s", label="after")
ax.axhline(1.0, color="black", lw=1.0, ls="--")
ax.set_xticks(columns)
ax.set_xlabel("initial flavour column")
ax.set_ylabel("column sum")
ax.set_title("Probability column normalization")
ax.legend()
fig.tight_layout()
save_and_show("diagnostic2_fig4_3_probability_normalization.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("matrix check before:", check_probability_matrix(P_perturbed))
print("matrix check after:", check_probability_matrix(P_repaired))

## 5. Flux

### 5.1 Flavour-Resolved Flux from Probabilities

A smooth toy spectrum is multiplied by energy-dependent probabilities and a scalar source normalization.

**Expected results**: at each energy, summing over final flavours recovers the scalar source flux times the spectrum because the probabilities are normalized.

In [ ]:
theta_E = 0.35 + 0.25 * torch.sin(torch.log(ENERGY_GRID))
P_E = probability_transition(rotation(theta_E))[..., :, 0]
spectrum = torch.exp(-ENERGY_GRID / 7.0) * ENERGY_GRID ** -0.4
flux_E = flux_from_probability(P_E, SOURCE_FLUX, spectrum)

fig, ax = plt.subplots(figsize=(7.8, 4.2))
for i, label in enumerate([r"$\nu_e$", r"$\nu_\mu$", r"$\nu_\tau$"]):
    ax.loglog(to_numpy(ENERGY_GRID), np.maximum(to_numpy(flux_E[:, i]), 1e-300), lw=1.8, label=label)
ax.loglog(to_numpy(ENERGY_GRID), to_numpy(SOURCE_FLUX * spectrum), color="black", lw=1.0, ls="--", label="total")
ax.set_xlabel("Energy [a.u.]")
ax.set_ylabel("flux [a.u.]")
ax.set_title("Flavour-resolved flux from final probabilities")
ax.legend()
fig.tight_layout()
save_and_show("diagnostic2_fig5_1_flux_spectrum.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max total-flux residual:", torch.max(torch.abs(flux_E.sum(dim=-1) - SOURCE_FLUX * spectrum)).item())

### 5.2 Source and Energy Broadcasting

The flux helper supports source and energy axes as long as the inputs are broadcastable with the leading probability dimensions.

**Expected results**: the output shape is `(n_source, n_energy, 3)`, and summing over flavours gives the source-by-energy normalization.

In [ ]:
source_flux = torch.tensor([2.0, 5.0], device=ctx.device, dtype=ctx.dtype)
source_spectrum = torch.stack([spectrum, 0.6 * spectrum], dim=0)
P_source_E = P_E.unsqueeze(0).expand(2, -1, -1)
flux_source_E = flux_from_probability(P_source_E, source_flux, source_spectrum)
expected_total = source_flux[:, None] * source_spectrum

fig, ax = plt.subplots(figsize=(7.8, 4.2))
for i in range(2):
    ax.loglog(to_numpy(ENERGY_GRID), to_numpy(flux_source_E[i].sum(dim=-1)), lw=1.8, label=f"source {i}")
ax.set_xlabel("Energy [a.u.]")
ax.set_ylabel("total flux [a.u.]")
ax.set_title("Broadcasted source and energy flux grids")
ax.legend()
fig.tight_layout()
save_and_show("diagnostic2_fig5_2_flux_broadcasting.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("flux grid shape:", tuple(flux_source_E.shape))
print("max broadcasting residual:", torch.max(torch.abs(flux_source_E.sum(dim=-1) - expected_total)).item())

## 6. Summary

The propagation/flux layer is deliberately generic. `evolutor.py` applies and composes already-built evolution operators, without knowing how they were generated. `probability.py` converts coherent amplitudes or incoherent weights into final flavour probabilities using the convention $P_{\beta\alpha}=|S_{\beta\alpha}|^2$. `flux.py` then applies source normalizations and optional spectra to those probabilities.

The plots show the expected behaviour for a unitary toy model: state probabilities remain normalized, segment composition preserves unitarity, transition matrices are column-normalized, and flavour-resolved fluxes sum back to the requested total flux normalization when the probabilities are normalized.